# NB_CREATE_DDL — 7 bảng control `lh_vv_ctrl.dbo`

Default lakehouse: **lh_vv_bronze**. Mọi tên bảng control viết đầy đủ `lh_vv_ctrl.dbo.<bảng>`.

**Thứ tự chạy:** tạo bảng, seed, rồi kiểm tra D1–D7, từ trên xuống.
Notebook này không DROP bảng nào. Tạo lại từ đầu và reset watermark chỉ làm theo runtime runbook.


## Ghi chú migration 08/10

DROP, sao lưu và Test Precheck đã bỏ khỏi notebook này. Notebook không DROP bảng nào.
Vị trí control cũ `lh_vv_bronze.ctrl` giữ làm mốc rollback. Việc xoá vị trí cũ là task riêng.


Tạo bảng ctrl_mng_pipeline_config


In [ ]:
%%sql
-- 1. CREATE TABLE

CREATE TABLE IF NOT EXISTS lh_vv_ctrl.dbo.ctrl_mng_pipeline_config (
    id          BIGINT,
    pl_name     STRING,
    src_schema  STRING,
    src_tbl     STRING,
    trg_schema  STRING,
    trg_tbl     STRING,
    is_active   INT,
    priority    INT,
    load_mode   STRING  COMMENT 'CDC (Debezium, partner) | DOC | DOC_ARRAY | LANG | LANG_ARRAY (snapshot, 3rd-party). NULL = CDC',
    align_path  STRING  COMMENT 'DOC_ARRAY / LANG_ARRAY: mảng tính từ gốc payload ghép theo vị trí, đọc bằng json_path _align.<trường>',
    dedup_order STRING  COMMENT 'Snapshot: chọn 1 dòng khi 1 tài liệu có nhiều dòng cùng khoá, vd is_original DESC, created_at DESC'
) USING DELTA;

-- 2. INSERT partner_raw_data
INSERT INTO lh_vv_ctrl.dbo.ctrl_mng_pipeline_config
    (id, pl_name, src_schema, src_tbl, trg_schema, trg_tbl, is_active, priority, load_mode, align_path, dedup_order)
SELECT
    COALESCE((SELECT MAX(id) FROM lh_vv_ctrl.dbo.ctrl_mng_pipeline_config), 0) + t.seq AS id,
    'PL_VV_TRANSFORM_BRONZE_TO_SILVER_FL_00'                          AS pl_name,
    'lh_vv_bronze.dbo'                                                AS src_schema,
    'partner_raw_data'                                                AS src_tbl,
    'lh_vv_silver.dbo'                                                AS trg_schema,
    t.trg_tbl,
    1                                                                 AS is_active,
    CASE WHEN t.trg_tbl LIKE 'slv_pn_product%' THEN 2 ELSE 1 END      AS priority,
    'CDC'                                                             AS load_mode,
    CAST(NULL AS STRING)                                              AS align_path,
    CAST(NULL AS STRING)                                              AS dedup_order
FROM VALUES
    ( 1, 'slv_pn_products'),
    ( 2, 'slv_pn_product_translations'),
    ( 3, 'slv_pn_product_attributes'),
    ( 4, 'slv_pn_product_attribute_translations'),
    ( 5, 'slv_pn_product_attribute_values'),
    ( 6, 'slv_pn_product_attribute_value_translations'),
    ( 7, 'slv_pn_product_variants'),
    ( 8, 'slv_pn_product_variant_translations'),
    ( 9, 'slv_pn_product_business_service_relation'),
    (10, 'slv_pn_product_posts'),
    (11, 'slv_pn_product_post_translations'),
    (12, 'slv_pn_partners'),
    (13, 'slv_pn_business_services'),
    (14, 'slv_pn_business_locations'),
    (15, 'slv_pn_hotel_room_types'),
    (16, 'slv_pn_hotel_rate_plans'),
    (17, 'slv_pn_business_service_i18ns'),
    (18, 'slv_pn_business_service_poi_link'),
    (19, 'slv_pn_orders'),
    (20, 'slv_pn_order_refs'),
    (21, 'slv_pn_order_item_tickets'),
    (22, 'slv_pn_order_item_hotels'),
    (23, 'slv_pn_order_item_flights')
AS t(seq, trg_tbl)
WHERE NOT EXISTS (
    SELECT 1 FROM lh_vv_ctrl.dbo.ctrl_mng_pipeline_config c
    WHERE c.pl_name = 'PL_VV_TRANSFORM_BRONZE_TO_SILVER_FL_00'
      AND c.src_schema = 'lh_vv_bronze.dbo' AND c.src_tbl = 'partner_raw_data'
      AND c.trg_schema = 'lh_vv_silver.dbo'
      AND c.trg_tbl = t.trg_tbl
);

-- 3. INSERT brz_3rd_crawler_poi_stream
INSERT INTO lh_vv_ctrl.dbo.ctrl_mng_pipeline_config
    (id, pl_name, src_schema, src_tbl, trg_schema, trg_tbl, is_active, priority, load_mode, align_path, dedup_order)
SELECT
    COALESCE((SELECT MAX(id) FROM lh_vv_ctrl.dbo.ctrl_mng_pipeline_config), 0) + t.seq AS id,
    'PL_VV_TRANSFORM_BRONZE_TO_SILVER_FL_00'                          AS pl_name,
    'lh_vv_bronze.dbo'                                                AS src_schema,
    'brz_3rd_crawler_poi_stream'                                                   AS src_tbl,
    'lh_vv_silver.dbo'                                                AS trg_schema,
    t.trg_tbl,
    t.is_active,                                                      -- [SỬA 09/10] 9 bảng; chỉ policy tắt
    1                                                                 AS priority,
    t.load_mode,
    t.align_path,
    t.dedup_order
FROM VALUES
    ( 1, 'slv_3p_poi'              , 'DOC'       , NULL        , NULL                               , 1),
    ( 2, 'slv_3p_poi_address'      , 'DOC'       , NULL        , NULL                               , 1),
    ( 3, 'slv_3p_poi_contact'      , 'DOC'       , NULL        , NULL                               , 1),
    ( 4, 'slv_3p_poi_opening_hours', 'DOC'       , NULL        , NULL                               , 1),
    ( 5, 'slv_3p_poi_policy'       , 'DOC'       , NULL        , NULL                               , 0),
    ( 6, 'slv_3p_poi_rating'       , 'DOC'       , NULL        , NULL                               , 1),
    ( 7, 'slv_3p_poi_amenity'      , 'DOC'       , NULL        , NULL                               , 1),
    ( 8, 'slv_3p_poi_review'       , 'DOC_ARRAY' , NULL        , NULL                               , 1),
    ( 9, 'slv_3p_poi_media'        , 'DOC_ARRAY' , NULL        , 'display_order ASC, media_id ASC'  , 1),
    (10, 'slv_3p_poi_price'        , 'DOC'       , NULL        , NULL                               , 1)
AS t(seq, trg_tbl, load_mode, align_path, dedup_order, is_active)
WHERE NOT EXISTS (
    SELECT 1 FROM lh_vv_ctrl.dbo.ctrl_mng_pipeline_config c
    WHERE c.pl_name = 'PL_VV_TRANSFORM_BRONZE_TO_SILVER_FL_00'
      AND c.src_schema = 'lh_vv_bronze.dbo' AND c.src_tbl = 'brz_3rd_crawler_poi_stream'
      AND c.trg_schema = 'lh_vv_silver.dbo'
      AND c.trg_tbl = t.trg_tbl
);


Reset silver (DROP/DELETE) không nằm trong luồng tạo bảng.

Cell tạo `ctrl_*` và seed registry chỉ CREATE / INSERT / `seed_registry`.
Không gộp `DROP TABLE`, `ALTER TABLE`, `DROP COLUMN` và `DELETE` trong cùng một cell.
Script xóa silver để nạp lại nằm ở `runtime-runbook.md` của task `20261009-redesign-3p-silver-raw-payload`.


In [ ]:
%%sql
-- [SỬA 09/10] Xóa hẳn dòng cấu hình của bảng đã bỏ. Không soft delete.
-- Bảng policy không nằm trong danh sách xóa.
-- Seed registry phía sau ghi lại 9 bảng còn lại (replaceWhere).

DELETE FROM lh_vv_ctrl.dbo.ctrl_cfg_schema_registry
WHERE src_schema = 'lh_vv_bronze.dbo'
  AND src_tbl = 'brz_3rd_crawler_poi_stream'
  AND trg_schema = 'lh_vv_silver.dbo'
  AND trg_tbl IN (
      'slv_3p_poi_raw_data',
      'slv_3p_poi_content',
      'slv_3p_poi_enrichment',
      'slv_3p_poi_review_i18n'
  );

DELETE FROM lh_vv_ctrl.dbo.ctrl_mng_pipeline_config
WHERE pl_name = 'PL_VV_TRANSFORM_BRONZE_TO_SILVER_FL_00'
  AND src_schema = 'lh_vv_bronze.dbo'
  AND src_tbl = 'brz_3rd_crawler_poi_stream'
  AND trg_schema = 'lh_vv_silver.dbo'
  AND trg_tbl IN (
      'slv_3p_poi_raw_data',
      'slv_3p_poi_content',
      'slv_3p_poi_enrichment',
      'slv_3p_poi_review_i18n'
  );

SELECT 'pipeline_config' AS bang, trg_tbl, load_mode, CAST(is_active AS STRING) AS is_active
FROM lh_vv_ctrl.dbo.ctrl_mng_pipeline_config
WHERE pl_name = 'PL_VV_TRANSFORM_BRONZE_TO_SILVER_FL_00'
  AND src_tbl = 'brz_3rd_crawler_poi_stream'
UNION ALL
SELECT 'schema_registry', trg_tbl, src_object, CAST(COUNT(*) AS STRING)
FROM lh_vv_ctrl.dbo.ctrl_cfg_schema_registry
WHERE src_tbl = 'brz_3rd_crawler_poi_stream'
GROUP BY trg_tbl, src_object
ORDER BY 1, 2;


Tạo bảng ctrl_mng_watermark


**[SỬA 03/10] Bỏ cell MERGE watermark từ `lh_vv_bronze.partner.brz_watermark`** (cell cũ dưới đây, không chạy):

```sql
MERGE INTO lh_vv_ctrl.dbo.ctrl_mng_watermark AS t
USING (SELECT watermark_id, watermark_value FROM lh_vv_bronze.partner.brz_watermark
       WHERE watermark_id = 'wm_transform_partner_raw_data') AS s
ON t.watermark_id = s.watermark_id
WHEN MATCHED THEN UPDATE SET t.watermark_value = s.watermark_value, t.updated_at = current_timestamp();
```

Lý do: cell nằm trước CREATE TABLE (bảng chưa có thì lỗi). Từ 02/10 notebook đọc raw theo `last_src_version`, `watermark_value` chỉ để theo dõi. Silver `slv_pn_*` tạo mới nên dòng partner phải ở trạng thái chưa chạy, không chép mốc cũ.


In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS lh_vv_ctrl.dbo.ctrl_mng_watermark (
    watermark_id        STRING,
    src_schema          STRING,
    src_tbl             STRING,
    trg_schema          STRING,
    trg_tbl             STRING,
    watermark_column    STRING,
    watermark_value     TIMESTAMP,
    last_success_at     TIMESTAMP,
    status              STRING,
    flow_name           STRING,
    watermark_sequence  BIGINT,
    last_run_id         STRING,
    last_src_version    BIGINT,
    last_src_table_id   STRING,
    error_message       STRING,
    lock_exec_id        STRING,
    lock_at             TIMESTAMP,
    updated_at          TIMESTAMP
)
USING DELTA;

INSERT INTO lh_vv_ctrl.dbo.ctrl_mng_watermark
    (watermark_id, src_schema, src_tbl, trg_schema, trg_tbl,
     watermark_column, watermark_value, last_success_at, status, flow_name,
     watermark_sequence, last_run_id, last_src_version, last_src_table_id, error_message,
     lock_exec_id, lock_at, updated_at)
SELECT s.*
FROM VALUES
    ('wm_transform_partner_raw_data', 'lh_vv_bronze.dbo', 'partner_raw_data', NULL, NULL,
     'EventProcessedUtcTime', NULL, NULL, 'INITIALIZED', 'NB_EXTRACT_PARTNER_CDC_BRZ_TO_SLV',
     -1, NULL, NULL, NULL, NULL,
     NULL, NULL, current_timestamp()),
    ('wm_transform_brz_3rd_crawler_poi_stream', 'lh_vv_bronze.dbo', 'brz_3rd_crawler_poi_stream', NULL, NULL,
     'crawled_at', NULL, NULL, 'INITIALIZED', 'NB_EXTRACT_3RD_PARTY_CDC_BRZ_TO_SLV',
     -1, NULL, NULL, NULL, NULL,
     NULL, NULL, current_timestamp())
AS s(watermark_id, src_schema, src_tbl, trg_schema, trg_tbl,
     watermark_column, watermark_value, last_success_at, status, flow_name,
     watermark_sequence, last_run_id, last_src_version, last_src_table_id, error_message,
     lock_exec_id, lock_at, updated_at)
WHERE NOT EXISTS (SELECT 1 FROM lh_vv_ctrl.dbo.ctrl_mng_watermark w WHERE w.watermark_id = s.watermark_id);


Tạo bảng ctrl_cfg_schema_registry 


In [ ]:
CONFIG_TABLE = "lh_vv_ctrl.dbo.ctrl_cfg_schema_registry"

_ddl = f""" 
CREATE TABLE IF NOT EXISTS lh_vv_ctrl.dbo.ctrl_cfg_schema_registry (
    src_schema          STRING    COMMENT 'Schema bảng raw, vd lh_vv_bronze.dbo',
    src_tbl             STRING    COMMENT 'Bảng raw, vd partner_raw_data (khớp ctrl_mng_pipeline_config.src_tbl)',
    src_object_schema   STRING    COMMENT 'CDC: schema trong Debezium source.schema, vd public. Snapshot (3rd-party): NULL',
    src_object          STRING    COMMENT 'CDC: bảng trong Debezium source.table, vd products. Snapshot: khối trong payload, vd poi_address ($ = gốc payload)',
    trg_schema          STRING    COMMENT 'Schema bảng đích, vd lh_vv_silver.dbo',
    trg_tbl             STRING    COMMENT 'Bảng đích, khớp ctrl_mng_pipeline_config.trg_tbl',
    trg_column          STRING    COMMENT 'Tên cột trong bảng đích',
    json_path           STRING    COMMENT 'CDC: đường dẫn trong JSON before/after. Snapshot: đường dẫn trong phần tử/khối, hoặc $ / _doc.cột / _root.a / _align.a / _lang / _pos / _langs. Nhiều phần p1,p2 chỉ với HASH_*',
    data_type           STRING    COMMENT 'Kiểu Spark của cột đích: string, bigint, int, smallint, boolean, timestamp, decimal(19,4)...',
    convert_rule        STRING    COMMENT 'Cách chuyển giá trị gốc: NONE | EPOCH_S_TS | EPOCH_MS_TS | EPOCH_US_TS | DATE_DAYS | DECIMAL_BASE64 | LOWER_TRIM. Khoá suy ra (snapshot): HASH_MD5_UUID | HASH_SHA256 | HASH_SHA256_PIPE',
    is_key              BOOLEAN   COMMENT 'Cột thuộc khoá MERGE',
    key_order           INT       COMMENT 'Thứ tự trong khoá ghép (1, 2...), NULL nếu không phải khoá',
    is_toast            BOOLEAN   COMMENT 'Cột có thể nhận giá trị TOAST thay thế của Debezium (chỉ CDC, snapshot luôn false)',
    column_order        INT       COMMENT 'Thứ tự cột trong bảng đích (1, 2...)',
    is_active           BOOLEAN   COMMENT 'Bật/tắt cột; bật/tắt cả bảng dùng ctrl_mng_pipeline_config.is_active',
    description         STRING    COMMENT 'Ghi chú'
) USING DELTA
COMMENT 'Mapping cột JSON sang bảng đích (CDC partner, snapshot 3rd-party) - 1 dòng / cột';
"""

print("--- DDL sap chay (kiem tra ky truoc khi execute) ---")
print(_ddl)
spark.sql(_ddl)

print(f"Đã tạo bảng config: {CONFIG_TABLE}")


In [ ]:


TABLE_CONFIGS = [
    {
        "source_table": "products",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_products",
        "key": "id",
        "columns": {
            "id":            {"path": "id",            "type": "string"},
            "external_id":   {"path": "external_id",   "type": "string"},
            "product_line":  {"path": "product_line",  "type": "string"},
            "status":        {"path": "status",        "type": "string"},
            "attributes":    {"path": "attributes",    "type": "string"},
            "metadata":      {"path": "metadata",      "type": "string"},
            "raw_sync_data": {"path": "raw_sync_data", "type": "string"},
            "created_at":    {"path": "created_at",    "type": "bigint"},
            "updated_at":    {"path": "updated_at",    "type": "bigint"},
            "deleted_at":    {"path": "deleted_at",    "type": "string"},
        },
        "toast_columns": ["attributes", "metadata", "raw_sync_data"],
    },
    {
        "source_table": "product_translations",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_product_translations",
        "key": "id",
        "columns": {
            "id":         {"path": "id",         "type": "bigint"},
            "product_id": {"path": "product_id", "type": "string"},
            "locale":     {"path": "locale",     "type": "string"},
            "name":       {"path": "name",       "type": "string"},
            "metadata":   {"path": "metadata",   "type": "string"},
            "created_at": {"path": "created_at", "type": "bigint"},
            "updated_at": {"path": "updated_at", "type": "bigint"},
        },
        "toast_columns": ["metadata"],
    },
    {
        "source_table": "product_attributes",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_product_attributes",
        "key": "id",
        "columns": {
            "id":                     {"path": "id",                     "type": "string"},
            "external_id":            {"path": "external_id",            "type": "string"},
            "code":                   {"path": "code",                   "type": "string"},
            "status":                 {"path": "status",                 "type": "string"},
            "product_category_scope": {"path": "product_category_scope", "type": "string"},
            "raw_sync_data":          {"path": "raw_sync_data",          "type": "string"},
            "created_at":             {"path": "created_at",             "type": "bigint"},
            "updated_at":             {"path": "updated_at",             "type": "bigint"},
        },
        "toast_columns": ["raw_sync_data"],
    },
    {
        "source_table": "product_attribute_translations",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_product_attribute_translations",
        "key": "id",
        "columns": {
            "id":           {"path": "id",           "type": "bigint"},
            "attribute_id": {"path": "attribute_id", "type": "string"},
            "locale":       {"path": "locale",       "type": "string"},
            "name":         {"path": "name",         "type": "string"},
            "created_at":   {"path": "created_at",   "type": "bigint"},
            "updated_at":   {"path": "updated_at",   "type": "bigint"},
        },
        "toast_columns": [],
    },
    {
        "source_table": "product_attribute_values",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_product_attribute_values",
        "key": "id",
        "columns": {
            "id":            {"path": "id",            "type": "string"},
            "attribute_id":  {"path": "attribute_id",  "type": "string"},
            "external_id":   {"path": "external_id",   "type": "string"},
            "code":          {"path": "code",          "type": "string"},
            "raw_sync_data": {"path": "raw_sync_data", "type": "string"},
            "created_at":    {"path": "created_at",    "type": "bigint"},
            "updated_at":    {"path": "updated_at",    "type": "bigint"},
        },
        "toast_columns": ["raw_sync_data"],
    },
    {
        "source_table": "product_attribute_value_translations",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_product_attribute_value_translations",
        "key": "id",
        "columns": {
            "id":           {"path": "id",           "type": "bigint"},
            "attribute_id": {"path": "attribute_id", "type": "string"},
            "locale":       {"path": "locale",       "type": "string"},
            "name":         {"path": "name",         "type": "string"},
        },
        "toast_columns": [],
    },
    {
        "source_table": "product_variants",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_product_variants",
        "key": "id",
        "columns": {
            "id":            {"path": "id",            "type": "string"},
            "product_id":    {"path": "product_id",    "type": "string"},
            "status":        {"path": "status",        "type": "string"},
            "external_id":   {"path": "external_id",   "type": "string"},
            "attributes":    {"path": "attributes",    "type": "string"},
            "raw_sync_data": {"path": "raw_sync_data", "type": "string"},
            "created_at":    {"path": "created_at",    "type": "bigint"},
            "updated_at":    {"path": "updated_at",    "type": "bigint"},
            "deleted_at":    {"path": "deleted_at",    "type": "string"},
        },
        "toast_columns": ["attributes", "raw_sync_data"],
    },
    {
        "source_table": "product_variant_translations",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_product_variant_translations",
        "key": "id",
        "columns": {
            "id":         {"path": "id",         "type": "bigint"},
            "variant_id": {"path": "variant_id", "type": "string"},
            "locale":     {"path": "locale",     "type": "string"},
            "name":       {"path": "name",       "type": "string"},
            "created_at": {"path": "created_at", "type": "bigint"},
            "updated_at": {"path": "updated_at", "type": "bigint"},
        },
        "toast_columns": [],
    },
    {
        "source_table": "product_business_service_relation",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_product_business_service_relation",
        "key": "id",
        "columns": {
            "id":                  {"path": "id",                  "type": "bigint"},
            "product_id":          {"path": "product_id",          "type": "string"},
            "business_service_id": {"path": "business_service_id", "type": "string"},
            "created_at":          {"path": "created_at",          "type": "bigint"},
            "updated_at":          {"path": "updated_at",          "type": "bigint"},
        },
        "toast_columns": [],
    },
    {
        "source_table": "product_posts",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_product_posts",
        "key": "id",
        "columns": {
            "id":                  {"path": "id",                  "type": "string"},
            "business_service_id": {"path": "business_service_id", "type": "string"},
            "created_by":          {"path": "created_by",          "type": "string"},
            "updated_by":          {"path": "updated_by",          "type": "string"},
            "post_code":           {"path": "post_code",           "type": "string"},
            "external_id":         {"path": "external_id",         "type": "string"},
            "status":              {"path": "status",              "type": "string"},
            "original_price":      {"path": "original_price",      "type": "string"},
            "markup_percent":      {"path": "markup_percent",      "type": "string"},
            "marketing_price":     {"path": "marketing_price",     "type": "string"},
            "thumbnail_url":       {"path": "thumbnail_url",       "type": "string"},
            "gallery":             {"path": "gallery",             "type": "string"},
            "product_list":        {"path": "product_list",        "type": "string"},
            "raw_sync_data":       {"path": "raw_sync_data",       "type": "string"},
            "validity_from":       {"path": "validity_from",       "type": "bigint"},
            "validity_to":         {"path": "validity_to",         "type": "bigint"},
            "created_at":          {"path": "created_at",          "type": "bigint"},
            "updated_at":          {"path": "updated_at",          "type": "bigint"},
            "deleted_at":          {"path": "deleted_at",          "type": "string"},
        },
        "toast_columns": [
            "original_price", "markup_percent", "marketing_price",
            "gallery", "product_list", "raw_sync_data",
        ],
    },
    {
        "source_table": "product_post_translations",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_product_post_translations",
        "key": "id",
        "columns": {
            "id":                {"path": "id",                "type": "bigint"},
            "product_post_id":   {"path": "product_post_id",   "type": "string"},
            "locale":            {"path": "locale",            "type": "string"},
            "title":             {"path": "title",             "type": "string"},
            "short_description": {"path": "short_description", "type": "string"},
            "tags":              {"path": "tags",              "type": "string"},
            "filter_tags":       {"path": "filter_tags",       "type": "string"},
            "ai_tags":           {"path": "ai_tags",           "type": "string"},
            "faqs":              {"path": "faqs",              "type": "string"},
            "sections":          {"path": "sections",          "type": "string"},
            "meta_title":        {"path": "meta_title",        "type": "string"},
            "meta_description":  {"path": "meta_description",  "type": "string"},
            "slug":              {"path": "slug",              "type": "string"},
            "created_at":        {"path": "created_at",        "type": "bigint"},
            "updated_at":        {"path": "updated_at",        "type": "bigint"},
        },
        "toast_columns": ["tags", "filter_tags", "ai_tags", "faqs", "sections"],
    },
    {
        "source_table": "partners",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_partners",
        "key": "id",
        "columns": {
            "id":                            {"path": "id",                            "type": "string"},
            "code":                          {"path": "code",                          "type": "string"},
            "lead_id":                       {"path": "lead_id",                       "type": "string"},
            "company_name":                  {"path": "company_name",                  "type": "string"},
            "company_short_name":            {"path": "company_short_name",            "type": "string"},
            "company_legal_name":            {"path": "company_legal_name",            "type": "string"},
            "company_legal_vietnamese_name": {"path": "company_legal_vietnamese_name", "type": "string"},
            "company_tax_id":                {"path": "company_tax_id",                "type": "string"},
            "company_registration_number":   {"path": "company_registration_number",   "type": "string"},
            "company_address":               {"path": "company_address",               "type": "string"},
            "business_sector":               {"path": "business_sector",               "type": "string"},
            "business_sector_detail":        {"path": "business_sector_detail",        "type": "string"},
            "detailed_profile":              {"path": "detailed_profile",              "type": "string"},
            "status":                        {"path": "status",                        "type": "string"},
            "assignee_username":             {"path": "assignee_username",             "type": "string"},
            "operation_status":              {"path": "operation_status",              "type": "string"},
            "created_at":                    {"path": "created_at",                    "type": "bigint"},
            "last_updated_at":               {"path": "last_updated_at",               "type": "bigint"},
            "created_by":                    {"path": "created_by",                    "type": "string"},
            "last_updated_by":               {"path": "last_updated_by",               "type": "string"},
            "tier":                          {"path": "tier",                          "type": "smallint"},
            "metadata":                      {"path": "metadata",                      "type": "string"},
            "parent_partner_id":             {"path": "parent_partner_id",             "type": "string"},
            "type":                          {"path": "type",                          "type": "string"},
            "business_scenario":             {"path": "business_scenario",             "type": "string"},
            "contact_information":           {"path": "contact_information",           "type": "string"},
            "business_registration_link":    {"path": "business_registration_link",    "type": "string"},
            "legal_representative":          {"path": "legal_representative",          "type": "string"},
            "business_registration_change":  {"path": "business_registration_change",  "type": "string"},
            "headquarter_information":       {"path": "headquarter_information",       "type": "string"},
        },
        "toast_columns": [
            "detailed_profile", "metadata", "contact_information",
            "legal_representative", "business_registration_change", "headquarter_information",
        ],
    },
    {
        "source_table": "business_services",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_business_services",
        "key": "id",
        "columns": {
            "id":                   {"path": "id",                   "type": "string"},
            "partner_id":           {"path": "partner_id",           "type": "string"},
            "business_location_id": {"path": "business_location_id", "type": "string"},
            "name":                 {"path": "name",                 "type": "string"},
            "phone":                {"path": "phone",                "type": "string"},
            "hotline":              {"path": "hotline",              "type": "string"},
            "email":                {"path": "email",                "type": "string"},
            "business_sector":      {"path": "business_sector",      "type": "string"},
            "operation_status":     {"path": "operation_status",     "type": "string"},
            "address":              {"path": "address",              "type": "string"},
            "direction_note":       {"path": "direction_note",       "type": "string"},
            "bio":                  {"path": "bio",                  "type": "string"},
            "story":                {"path": "story",                "type": "string"},
            "usps":                 {"path": "usps",                 "type": "string"},
            "local_tips":           {"path": "local_tips",           "type": "string"},
            "media":                {"path": "media",                "type": "string"},
            "is_main":              {"path": "is_main",              "type": "boolean"},
            "amentities":           {"path": "amentities",           "type": "string"},
            "created_at":           {"path": "created_at",           "type": "bigint"},
            "last_updated_at":      {"path": "last_updated_at",      "type": "bigint"},
            "created_by":           {"path": "created_by",           "type": "string"},
            "last_updated_by":      {"path": "last_updated_by",      "type": "string"},
            "metadata":             {"path": "metadata",             "type": "string"},
            "is_active":            {"path": "is_active",            "type": "boolean"},
            "last_in_activated_at": {"path": "last_in_activated_at", "type": "bigint"},
            "last_in_activated_by": {"path": "last_in_activated_by", "type": "string"},
            "website_urls":         {"path": "website_urls",         "type": "string"},
            "last_verify_at":       {"path": "last_verify_at",       "type": "epoch_seconds_timestamp"},
        },
        "toast_columns": [
            "address", "direction_note", "bio", "story", "usps",
            "local_tips", "media", "amentities", "metadata", "website_urls",
        ],
    },
    {
        "source_table": "business_locations",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_business_locations",
        "key": "id",
        "columns": {
            "id":                {"path": "id",                "type": "string"},
            "name":              {"path": "name",              "type": "string"},
            "type":              {"path": "type",              "type": "string"},
            "address":           {"path": "address",           "type": "string"},
            "direction_note":    {"path": "direction_note",    "type": "string"},
            "partner_id":        {"path": "partner_id",        "type": "string"},
            "business_scenario": {"path": "business_scenario", "type": "string"},
            "created_at":        {"path": "created_at",        "type": "bigint"},
            "last_updated_at":   {"path": "last_updated_at",   "type": "bigint"},
            "last_updated_by":   {"path": "last_updated_by",   "type": "string"},
            "metadata":          {"path": "metadata",          "type": "string"},
            "created_by":        {"path": "created_by",        "type": "string"},
        },
        "toast_columns": ["address", "direction_note", "metadata"],
    },
    {
        "source_table": "hotel_room_types",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_hotel_room_types",
        "key": "id",
        "columns": {
            "id":                   {"path": "id",                   "type": "string"},
            "code":                 {"path": "code",                 "type": "string"},
            "name":                 {"path": "name",                 "type": "string"},
            "description":          {"path": "description",          "type": "string"},
            "status":               {"path": "status",               "type": "string"},
            "max_occupancy":        {"path": "max_occupancy",        "type": "int"},
            "adult_capacity":       {"path": "adult_capacity",       "type": "int"},
            "child_capacity":       {"path": "child_capacity",       "type": "int"},
            "max_extra_bed":        {"path": "max_extra_bed",        "type": "int"},
            "max_child_surcharge":  {"path": "max_child_surcharge",  "type": "int"},
            "room_size":            {"path": "room_size",            "type": "int"},
            "view_type":            {"path": "view_type",            "type": "string"},
            "total_physical_rooms": {"path": "total_physical_rooms", "type": "int"},
            "total_pseudo_rooms":   {"path": "total_pseudo_rooms",   "type": "int"},
            "out_of_service_count": {"path": "out_of_service_count", "type": "int"},
            "out_of_order_count":   {"path": "out_of_order_count",   "type": "int"},
            "total_rooms":          {"path": "total_rooms",          "type": "int"},
            "available_rooms":      {"path": "available_rooms",      "type": "int"},
            "amenities":            {"path": "amenities",            "type": "string"},
            "media_files":          {"path": "media_files",          "type": "string"},
            "tags":                 {"path": "tags",                 "type": "string"},
            "business_service_id":  {"path": "business_service_id",  "type": "string"},
            "created_at":           {"path": "created_at",           "type": "bigint"},
            "updated_at":           {"path": "updated_at",           "type": "bigint"},
            "deleted_at":           {"path": "deleted_at",           "type": "string"},
        },
        "toast_columns": ["description", "amenities", "media_files", "tags"],
    },
    {
        "source_table": "hotel_rate_plans",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_hotel_rate_plans",
        "key": "id",
        "columns": {
            "id":                     {"path": "id",                     "type": "string"},
            "business_service_id":    {"path": "business_service_id",    "type": "string"},
            "code":                   {"path": "code",                   "type": "string"},
            "name":                   {"path": "name",                   "type": "string"},
            "description":            {"path": "description",            "type": "string"},
            "status":                 {"path": "status",                 "type": "string"},
            "kind":                   {"path": "kind",                   "type": "string"},
            "meal_plan":              {"path": "meal_plan",              "type": "int"},
            "include_tax":            {"path": "include_tax",            "type": "boolean"},
            "include_service_charge": {"path": "include_service_charge", "type": "boolean"},
            "is_derived":             {"path": "is_derived",             "type": "boolean"},
            "valid_from":             {"path": "valid_from",             "type": "bigint"},
            "valid_to":               {"path": "valid_to",               "type": "bigint"},
            "is_visible_on_website":  {"path": "is_visible_on_website",  "type": "boolean"},
            "tags":                   {"path": "tags",                   "type": "string"},
            "cancellation_policies":  {"path": "cancellation_policies",  "type": "string"},
            "created_at":             {"path": "created_at",             "type": "bigint"},
            "updated_at":             {"path": "updated_at",             "type": "bigint"},
            "deleted_at":             {"path": "deleted_at",             "type": "string"},
        },
        "toast_columns": ["description", "tags", "cancellation_policies"],
    },
    {
        "source_table": "business_service_i18ns",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_business_service_i18ns",
        "key": ["business_service_id", "locale"],
        "columns": {
            "business_service_id": {"path": "business_service_id", "type": "string"},
            "locale":              {"path": "locale",              "type": "string"},
            "name":                {"path": "name",                "type": "string"},
            "bio":                 {"path": "bio",                 "type": "string"},
            "story":               {"path": "story",               "type": "string"},
            "direction_note":      {"path": "direction_note",      "type": "string"},
            "usps":                {"path": "usps",                "type": "string"},
            "local_tips":          {"path": "local_tips",          "type": "string"},
            "amentities":          {"path": "amentities",          "type": "string"},
            "media":               {"path": "media",               "type": "string"},
            "created_at":          {"path": "created_at",          "type": "bigint"},
            "updated_at":          {"path": "updated_at",          "type": "bigint"},
        },
        "toast_columns": [
            "name", "bio", "story", "direction_note",
            "usps", "local_tips", "amentities", "media",
        ],
    },
    {
        "source_table": "business_service_poi_link",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_business_service_poi_link",
        "key": "id",
        "columns": {
            "id":                   {"path": "id",                   "type": "bigint"},
            "business_service_id":  {"path": "business_service_id",  "type": "string"},
            "poi_id":               {"path": "poi_id",               "type": "string"},
            "created_at":           {"path": "created_at",           "type": "bigint"},
            "updated_at":           {"path": "updated_at",           "type": "bigint"},
            "lang":                 {"path": "lang",                 "type": "string"},
        },
        "toast_columns": [],
    },
    {
        "source_table": "orders",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_orders",
        "key": "id",
        "columns": {
            "id":             {"path": "id",             "type": "string"},
            "code":           {"path": "code",           "type": "string"},
            "contact_info":   {"path": "contact_info",   "type": "string"},
            "user_id":        {"path": "user_id",        "type": "string"},
            "total_payment":  {"path": "total_payment",  "type": "debezium_decimal(19,4)"},
            "currency":       {"path": "currency",       "type": "string"},
            "status":         {"path": "status",         "type": "string"},
            "payment_status": {"path": "payment_status", "type": "string"},
            "payment_id":     {"path": "payment_id",     "type": "string"},
            "metadata":       {"path": "metadata",       "type": "string"},
            "expire_at":      {"path": "expire_at",      "type": "bigint"},
            "created_at":     {"path": "created_at",     "type": "bigint"},
            "updated_at":     {"path": "updated_at",     "type": "bigint"},
        },
        "toast_columns": ["contact_info", "metadata"],
    },
    {
        "source_table": "order_refs",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_order_refs",
        "key": "id",
        "columns": {
            "id":                    {"path": "id",                    "type": "string"},
            "order_id":              {"path": "order_id",              "type": "string"},
            "status":                {"path": "status",                "type": "string"},
            "total_payment":         {"path": "total_payment",         "type": "debezium_decimal(19,4)"},
            "sub_total":             {"path": "sub_total",             "type": "debezium_decimal(19,4)"},
            "external_order_id":     {"path": "external_order_id",     "type": "string"},
            "internal_input_object": {"path": "internal_input_object", "type": "string"},
            "product_type":          {"path": "product_type",          "type": "string"},
            "raw_data":              {"path": "raw_data",              "type": "string"},
            "created_at":            {"path": "created_at",            "type": "bigint"},
            "updated_at":            {"path": "updated_at",            "type": "bigint"},
        },
        "toast_columns": ["internal_input_object", "raw_data"],
    },
    {
        "source_table": "order_item_tickets",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_order_item_tickets",
        "key": "order_ref_id",
        "columns": {
            "order_id":                     {"path": "order_id",                     "type": "string"},
            "order_ref_id":                 {"path": "order_ref_id",                 "type": "string"},
            "product_id":                   {"path": "product_id",                   "type": "string"},
            "variant_id":                   {"path": "variant_id",                   "type": "string"},
            "provider":                     {"path": "provider",                     "type": "string"},
            "pnr":                          {"path": "pnr",                          "type": "string"},
            "ticket_number":                {"path": "ticket_number",                "type": "string"},
            "verify_code":                  {"path": "verify_code",                  "type": "string"},
            "external_ticket_product_name": {"path": "external_ticket_product_name", "type": "string"},
            "usage_date":                   {"path": "usage_date",                   "type": "timestamp"},
            "valid_from":                   {"path": "valid_from",                   "type": "timestamp"},
            "valid_to":                     {"path": "valid_to",                     "type": "timestamp"},
            "raw_data":                     {"path": "raw_data",                     "type": "string"},
            "created_at":                   {"path": "created_at",                   "type": "bigint"},
            "updated_at":                   {"path": "updated_at",                   "type": "bigint"},
        },
        "toast_columns": ["raw_data"],
    },
    {
        "source_table": "order_item_hotels",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_order_item_hotels",
        "key": "order_id",
        "columns": {
            "order_id":       {"path": "order_id",       "type": "string"},
            "order_ref_id":   {"path": "order_ref_id",   "type": "string"},
            "check_in_date":  {"path": "check_in_date",  "type": "timestamp"},
            "checkout_date":  {"path": "checkout_date",  "type": "timestamp"},
            "number_of_room": {"path": "number_of_room", "type": "smallint"},
            "hotel_id":       {"path": "hotel_id",       "type": "string"},
            "pnr":            {"path": "pnr",            "type": "string"},
            "room_type_id":   {"path": "room_type_id",   "type": "string"},
            "rate_plan_id":   {"path": "rate_plan_id",   "type": "string"},
            "raw_data":       {"path": "raw_data",       "type": "string"},
            "created_at":     {"path": "created_at",     "type": "bigint"},
            "updated_at":     {"path": "updated_at",     "type": "bigint"},
        },
        "toast_columns": ["raw_data"],
    },
    {
        "source_table": "order_item_flights",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_order_item_flights",
        "key": "order_id",
        "columns": {
            "order_id":           {"path": "order_id",           "type": "string"},
            "order_ref_id":       {"path": "order_ref_id",       "type": "string"},
            "provider":           {"path": "provider",           "type": "string"},
            "pnr":                {"path": "pnr",                "type": "string"},
            "booking_id":         {"path": "booking_id",         "type": "string"},
            "segment_id":         {"path": "segment_id",         "type": "smallint"},
            "carrier_code":       {"path": "carrier_code",       "type": "string"},
            "carrier_name":       {"path": "carrier_name",       "type": "string"},
            "departure":          {"path": "departure",          "type": "string"},
            "departure_time":     {"path": "departure_time",     "type": "timestamp"},
            "departure_terminal": {"path": "departure_terminal", "type": "string"},
            "arrival":            {"path": "arrival",            "type": "string"},
            "arrival_time":       {"path": "arrival_time",       "type": "timestamp"},
            "arrival_terminal":   {"path": "arrival_terminal",   "type": "string"},
            "flight_number":      {"path": "flight_number",      "type": "string"},
            "booking_class":      {"path": "booking_class",      "type": "string"},
            "aircraft":           {"path": "aircraft",           "type": "string"},
            "fare_basis_code":    {"path": "fare_basis_code",    "type": "string"},
            "fare_basis_detail":  {"path": "fare_basis_detail",  "type": "string"},
            "raw_data":           {"path": "raw_data",           "type": "string"},
            "created_at":         {"path": "created_at",         "type": "bigint"},
            "updated_at":         {"path": "updated_at",         "type": "bigint"},
        },
        "toast_columns": ["fare_basis_detail", "raw_data"],
    },
]


In [ ]:
# =============================================================================
# Seed ctrl_cfg_schema_registry (v2) từ TABLE_CONFIGS
# - Chỉ thay dữ liệu của 1 nguồn (replaceWhere) -> không đụng cấu hình nguồn khác
# - Chạy lại bao nhiêu lần cũng cho cùng kết quả
# Cần có biến TABLE_CONFIGS (23 bảng partner) ở cell trước.
# =============================================================================
from pyspark.sql.types import (
    BooleanType, IntegerType, StringType, StructField, StructType,
)

REGISTRY_TABLE = "lh_vv_ctrl.dbo.ctrl_cfg_schema_registry"
SRC_SCHEMA = "lh_vv_bronze.dbo"
SRC_TBL = "partner_raw_data"

import re

# Kiểu giả trong TABLE_CONFIGS -> (kiểu Spark thật, convert_rule)
PSEUDO_TYPES = {
    "epoch_seconds_timestamp": ("timestamp", "EPOCH_S_TS"),
}
# Decimal Debezium (decimal.handling.mode = precise): JSON là base64 của số nguyên chưa chia scale
DEBEZIUM_DECIMAL_RE = re.compile(r"debezium_decimal\(\s*(\d+)\s*,\s*(\d+)\s*\)", re.I)


def resolve_type(t: str) -> tuple:
    """Kiểu trong TABLE_CONFIGS -> (data_type, convert_rule).
    "debezium_decimal(19,4)"  -> ("decimal(19,4)", "DECIMAL_BASE64")
    "epoch_seconds_timestamp" -> ("timestamp", "EPOCH_S_TS")
    kiểu khác                 -> (t, "NONE")
    """
    t = t.strip()
    m = DEBEZIUM_DECIMAL_RE.fullmatch(t)
    if m:
        p, s = int(m.group(1)), int(m.group(2))
        if not (0 < p <= 38 and 0 <= s <= p):
            raise ValueError(f"debezium_decimal sai precision/scale: {t}")
        return f"decimal({p},{s})", "DECIMAL_BASE64"
    return PSEUDO_TYPES.get(t, (t, "NONE"))


KNOWN_RULES = {"NONE", "EPOCH_S_TS", "EPOCH_MS_TS", "EPOCH_US_TS", "DATE_DAYS", "DECIMAL_BASE64", "LOWER_TRIM",
               "HASH_MD5_UUID", "HASH_SHA256", "HASH_SHA256_PIPE"}

REGISTRY_SCHEMA = StructType([
    StructField("src_schema",        StringType(),  False),
    StructField("src_tbl",           StringType(),  False),
    StructField("src_object_schema", StringType(),  True),
    StructField("src_object",        StringType(),  False),
    StructField("trg_schema",        StringType(),  False),
    StructField("trg_tbl",           StringType(),  False),
    StructField("trg_column",        StringType(),  False),
    StructField("json_path",         StringType(),  False),
    StructField("data_type",         StringType(),  False),
    StructField("convert_rule",      StringType(),  False),
    StructField("is_key",            BooleanType(), False),
    StructField("key_order",         IntegerType(), True),
    StructField("is_toast",          BooleanType(), False),
    StructField("column_order",      IntegerType(), False),
    StructField("is_active",         BooleanType(), False),
    StructField("description",       StringType(),  True),
])


def flatten_table_configs(configs: list, src_schema: str, src_tbl: str) -> list:
    """TABLE_CONFIGS (1 phần tử / bảng) -> danh sách dòng (1 dòng / cột đích)."""
    rows = []
    for cfg in configs:
        keys = cfg["key"] if isinstance(cfg["key"], list) else [cfg["key"]]
        toast_cols = set(cfg.get("toast_columns", []))

        for col_order, (col_name, col_cfg) in enumerate(cfg["columns"].items(), start=1):
            data_type, convert_rule = resolve_type(col_cfg["type"])
            convert_rule = col_cfg.get("rule", convert_rule)
            rows.append({
                "src_schema":        src_schema,
                "src_tbl":           src_tbl,
                "src_object_schema": cfg.get("source_schema"),
                "src_object":        cfg["source_table"],
                "trg_schema":        cfg["target_schema"],
                "trg_tbl":           cfg["target_table"],
                "trg_column":        col_name,
                "json_path":         col_cfg["path"],
                "data_type":         data_type,
                "convert_rule":      convert_rule,
                "is_key":            col_name in keys,
                "key_order":         keys.index(col_name) + 1 if col_name in keys else None,
                "is_toast":          col_name in toast_cols,
                "column_order":      col_order,
                "is_active":         bool(col_cfg.get("active", True)),
                "description":       None,
            })
    return rows


def validate_rows(configs: list, rows: list) -> None:
    """Dừng ngay nếu cấu hình sai, trước khi ghi."""
    errors = []

    seen = set()
    for r in rows:
        k = (r["trg_schema"], r["trg_tbl"], r["trg_column"])
        if k in seen:
            errors.append(f"Trùng cột: {k}")
        seen.add(k)

    for cfg in configs:
        tbl = cfg["target_table"]
        keys = cfg["key"] if isinstance(cfg["key"], list) else [cfg["key"]]
        missing_keys = [k for k in keys if k not in cfg["columns"]]
        missing_toast = [c for c in cfg.get("toast_columns", []) if c not in cfg["columns"]]
        if missing_keys:
            errors.append(f"{tbl}: khoá không có trong columns {missing_keys}")
        if missing_toast:
            errors.append(f"{tbl}: toast_columns không có trong columns {missing_toast}")

    # [THÊM 03/10 - 3P] tên convert_rule
    errors += [f"{r['trg_tbl']}.{r['trg_column']}: convert_rule không hỗ trợ {r['convert_rule']}"
               for r in rows if r["convert_rule"] not in KNOWN_RULES]
    # [THÊM 04/10 - 3P] cột khoá không được tắt
    errors += [f"{r['trg_tbl']}.{r['trg_column']}: cột khoá không được tắt (active False)"
               for r in rows if r["is_key"] and not r["is_active"]]

    if errors:
        raise ValueError("Cấu hình TABLE_CONFIGS sai:\n  - " + "\n  - ".join(errors))


# [SỬA 03/10 - 3P] gói thành hàm (logic như cũ) để dùng cho cả nguồn 3rd-party
def seed_registry(configs: list, src_schema: str, src_tbl: str) -> int:
    """Thay toàn bộ dòng registry của 1 nguồn (replaceWhere) bằng cấu hình configs. Trả số cột đã ghi."""
    rows = flatten_table_configs(configs, src_schema, src_tbl)
    validate_rows(configs, rows)
    (
        spark.createDataFrame(rows, schema=REGISTRY_SCHEMA)
        .write.format("delta")
        .mode("overwrite")
        .option("replaceWhere", f"src_schema = '{src_schema}' AND src_tbl = '{src_tbl}'")
        .saveAsTable(REGISTRY_TABLE)
    )
    print(f"Seed {len(rows):,} cột / {len(configs)} bảng -> {REGISTRY_TABLE} (nguồn {src_schema}.{src_tbl})")
    return len(rows)


def show_registry(src_schema: str, src_tbl: str) -> None:
    """[THÊM 03/10] Kiểm tra sau seed: số cột, khoá, cột có convert_rule khác NONE; [3P] kèm load_mode từ pipeline_config."""
    display(spark.sql(f"""
        SELECT r.trg_tbl, MAX(p.load_mode) AS load_mode, MAX(r.src_object) AS src_object,
               COUNT(*) AS so_cot,
               SUM(CASE WHEN r.is_active THEN 0 ELSE 1 END) AS so_cot_tat,      -- [THÊM 04/10 - 3P]
               MAX(p.is_active) AS bang_active,                                   -- [THÊM 04/10 - 3P]
               array_join(array_sort(collect_list(CASE WHEN r.is_key THEN r.trg_column END)), ', ') AS khoa,
               array_join(array_sort(collect_list(CASE WHEN r.convert_rule <> 'NONE'
                                                       THEN concat(r.trg_column, ':', r.convert_rule) END)), ', ')
                   AS convert_rule_khac_none
        FROM {REGISTRY_TABLE} r
        LEFT JOIN lh_vv_ctrl.dbo.ctrl_mng_pipeline_config p
          ON p.src_schema = r.src_schema AND p.src_tbl = r.src_tbl AND p.trg_schema = r.trg_schema AND p.trg_tbl = r.trg_tbl
        WHERE r.src_schema = '{src_schema}' AND r.src_tbl = '{src_tbl}'
        GROUP BY r.trg_tbl
        ORDER BY r.trg_tbl
    """))


seed_registry(TABLE_CONFIGS, SRC_SCHEMA, SRC_TBL)
show_registry(SRC_SCHEMA, SRC_TBL)


**[THÊM 03/10 - 3P] Seed registry nguồn 3rd-party `brz_3rd_crawler_poi_stream`** — 14 bảng `slv_3p_poi_*` ([SỬA 04/10] bảng policy tắt), notebook `NB_EXTRACT_3RD_PARTY_CDC_BRZ_TO_SLV`.

Chạy sau cell seed partner (dùng lại `seed_registry`, `show_registry`). `load_mode` / `align_path` / `dedup_order` của từng bảng nằm ở cell `ctrl_mng_pipeline_config` (mục 3).
Thêm/bớt cột: sửa `TABLE_CONFIGS_3P` rồi chạy lại 2 cell dưới (chỉ thay dòng của `brz_3rd_crawler_poi_stream`), hoặc `INSERT` thẳng 1 dòng vào registry. Notebook tự `ADD COLUMNS` ở lần chạy sau.


In [ ]:
# [THÊM 03/10 - 3P]
# =============================================================================
# [THÊM 03/10] TABLE_CONFIGS_3P — bảng slv_3p_poi_* của nguồn 3rd-party (lh_vv_bronze.dbo.brz_3rd_crawler_poi_stream)
# Notebook xử lý: NB_EXTRACT_3RD_PARTY_CDC_BRZ_TO_SLV. Thiết kế: claude/POI_3P_EXTRACT_REDESIGN.md §5.4, §5.8
#
# [SỬA 04/10 - 3P] Đối chiếu payload thật (truy vấn A, E; VERIFY_3P_CONFIG_0410 V1–V13 trên 21.168 dòng raw, crawl
#   tới 16/09) + quyết định của anh Hòa 04/10. 14 bảng (bảng policy tắt ở pipeline_config).
#   - Cột = khoá có trong payload. Bỏ cột của khoá không tồn tại (V2: dữ liệu bảng cũ toàn NULL, trừ primary_type — xem D2).
#   - D1  Khoá luôn null (V1) -> "active": False: registry is_active = false, notebook bỏ qua, bảng đích không có cột.
#         Collector bắt đầu điền: UPDATE registry is_active = true rồi full_reload bảng đó. Bảng policy (khối luôn null):
#         tắt ở ctrl_mng_pipeline_config.is_active = 0.
#   - D2  Bỏ cột JSON nguyên khối trùng các cột đã tách (address_json, contact_json, amenity_json, opening_hours_json,
#         rating_json). Giữ nguyên JSON, transform sau: raw_data_json, amenity_schema_json, ext_attributes_json,
#         facilities_json, secondary_hours_json (anh Hòa chốt 04/10 15:53: amenity_schema / ext_attributes không tách
#         key-value ở extract -> không cần load_mode DOC_MAP, code lib / notebook giữ nguyên bản 04/10 sáng).
#   - D3  [SỬA 09/10] open_now của poi_opening_hours thành cột boolean. ext_attributes.open_now vẫn nằm trong JSON.
#   - D4  Content: khoá (poi_id, locale, content_type), trùng khoá lấy phần tử đầu (V8: 29 nhóm trùng, 0 nhóm khác nội dung).
#   - D5  Bỏ cột trùng 100% với cột khác (V4, V5, V12): slv_3p_poi.price_level (= poi_price.price_level),
#         slv_3p_poi.country_code (= poi_address.country_code), slv_3p_poi.source (= source_name viết hoa),
#         address.address_google (= full_address), enrichment.time_to_spend (= time_to_spend_minutes). Giữ enrichment_source_id.
#   - D7  Mỗi cột có comment ý nghĩa. "(nghĩa chưa xác nhận)": suy từ tên / mẫu, chưa có tài liệu collector.
#   - Mảng 1 tầng để cột JSON (anh Hòa chốt 04/10 15:53): periods_json, experiences_json, types_json,
#     subcategory_tags_json, weekday_text_json. Cần tách sau: chỉ thêm cấu hình DOC_ARRAY / LANG_ARRAY, không sửa code.
#   Từng chỗ: [THÊM 04/10] / [BỎ 04/10] / [TẮT 04/10 D1] / [SỬA 04/10].
#
# [SỬA 09/10] Payload raw, probe E1 (33/33 tài liệu, extra_info rỗng, không enrichment).
#   L1 chỉ extract 1:1 và cast kiểu. Quyết định nghiệp vụ ở L2.
#   Bỏ AMENITY_FLAGS, available_langs_json, enrichment_failed_langs_json.
#   Bỏ khỏi registry: slv_3p_poi_price, slv_3p_poi_raw_data, slv_3p_poi_content,
#   slv_3p_poi_enrichment, slv_3p_poi_review_i18n. slv_3p_poi_policy vẫn tắt ở pipeline_config.
#   Thêm lại slv_3p_poi_price (10/10, 13/53 tài liệu là object đủ 4 khóa). Kỳ vọng: 10 bảng, 98 cột, 15 cột tắt.
#
# Mỗi bảng:
#   source_table : src_object = khối trong payload ("$" = gốc payload; bảng LANG*: đường dẫn trong object bản địa hoá)
#   (load_mode / align_path / dedup_order / is_active của bảng: cell ctrl_mng_pipeline_config, mục 3)
#   key          : cột khoá (str hoặc list theo thứ tự)
#   columns      : {tên cột: {"path": json_path, "type": kiểu Spark, "rule": convert_rule (mặc định NONE),
#                             "active": False nếu tắt (mặc định bật)}}
#
# json_path (bảng ở NB_LIB_EXTRACT_RAWDATA §1):
#   a.b           trường trong phần tử / khối (bảng DOC "$": trong gốc payload)
#   $             cả phần tử / khối dạng chuỗi JSON
#   _doc.<cột>    cột của bảng raw            _pos    vị trí phần tử (từ 0)
#   [BỎ 09/10] _lang, _langs, _align: 3P không còn bảng LANG / LANG_ARRAY.
#   p1,p2         nhiều phần, chỉ với rule HASH_*
#
# Quy ước: cột chứa JSON (object/mảng) giữ dạng chuỗi, tên có hậu tố _json (quyết định mở #4: JSON STRING).
#          Tên cột giữ theo tên trường payload (snake_case), trừ trường camelCase của enrichment.
# Kiểu: V1 04/10 CAST mọi cột 0 lỗi (thời gian ISO 8601 9 chữ số lẻ cắt về microgiây).
# =============================================================================
_S = "lh_vv_silver.dbo"
# Khoá địa điểm: md5(source_name + source_id) dạng UUID = poi_group_id của luồng cũ (quyết định 03/10)
POI_ID = {"path": "_doc.source_name,_doc.source_id", "type": "string", "rule": "HASH_MD5_UUID"}


def _str(path):
    return {"path": path, "type": "string"}


def _bool(path):
    return {"path": path, "type": "boolean"}


def _off(col):
    """[THÊM 04/10 D1] Cột tắt (registry is_active = false): khoá có trong payload nhưng luôn null (V1 04/10)."""
    return {**col, "active": False}


# [BỎ 09/10] AMENITY_FLAGS / AMENITY_FLAGS_OFF: E1 amenity_flat_flag_docs = 0.
#   Map ext_attributes sang cờ nghiệp vụ là logic L2, không làm ở extract.

TABLE_CONFIGS_3P = [
    # ------------------------------------------------------------------ POI: 1 dòng / địa điểm, trường cấp 1
    {
        "source_table": "$", "target_schema": _S, "target_table": "slv_3p_poi",
        "key": "poi_id",
        "columns": {
            "poi_id":               POI_ID,
            "source_name":          _str("_doc.source_name"),     # nguồn crawl (cột raw), vd google
            "source_id":            _str("_doc.source_id"),       # id địa điểm ở nguồn (cột raw), vd Google place id ChIJ…
            "language_code":        {"path": "_doc.language_code", "type": "string", "rule": "LOWER_TRIM"},
                                                                  # ngôn ngữ của tài liệu (cột raw), vd vi; V4a: = payload
            # [BỎ 09/10] available_langs_json (_langs): suy từ enrichment, không phải trường 1:1
            "poi_name":             _str("poi_name"),             # tên địa điểm
            "poi_name_normalized":  _str("poi_name_normalized"),  # tên viết thường (quy tắc chuẩn hoá chưa xác nhận)
            "business_sector":      _str("business_sector"),      # ngành, vd f_n_b, accommodation (V6b: có cả
                                                                  # 'accomodation' viết sai, 250 dòng)
            "business_category":    _str("business_category"),    # nhóm trong ngành, vd other (nghĩa chưa xác nhận)
            "types_json":           _str("types"),                # mảng loại địa điểm kiểu Google (≤ 20, 386 giá trị)
            "operating_status":     _str("operating_status"),     # trạng thái hoạt động, vd operational
            "slug":                 _str("slug"),                 # chuỗi định danh cho URL (5.400 / 21.168 có)
            "partner_id":           _off(_str("partner_id")),     # [TẮT 04/10 D1] id đối tác (nghĩa chưa xác nhận)
            # [BỎ 04/10 D5] source (= source_name viết hoa), country_code (= poi_address.country_code),
            #               price_level (= poi_price.price_level)
            "collector_poi_id":     _str("poi_id"),               # [THÊM 04/10] UUID địa điểm của collector; V9: 1-1
                                                                  # với (source_name, source_id), khác poi_id md5
            "subcategory_tags_json": _str("subcategory_tags"),    # [THÊM 04/10] mảng tag phân loại con (≤ 20, 399 giá trị;
                                                                  # quan hệ với types chưa xác nhận)
            "facilities_json":      _str("facilities"),           # [THÊM 04/10] JSON nhiều tầng dạng schema tiện ích
                                                                  # (5.771 giá trị khác nhau, V12: khác amenity_schema.
                                                                  # facilities ở 1.805 dòng; nghĩa chưa xác nhận). Giữ khối
            "source_created_at":    {"path": "created_at", "type": "timestamp"},
                                                                  # [THÊM 04/10] created_at của payload (nghĩa chưa xác nhận)
            # [BỎ 09/10] enrichment_failed_langs_json: E1 không còn extra_info.enrichmentFailedLangs
            "accommodation_type":   _off(_str("accommodation_type")),      # [TẮT 04/10 D1] loại hình lưu trú
            "star_rating":          _off(_str("star_rating")),             # [TẮT 04/10 D1] hạng sao
            "awards_json":          _off(_str("awards")),                  # [TẮT 04/10 D1] giải thưởng
            "slug_history_json":    _off(_str("slug_history")),            # [TẮT 04/10 D1] các slug cũ
            "poi_amenity_schema_json": _off(_str("poi_amenity_schema")),   # [TẮT 04/10 D1] (nghĩa chưa xác nhận)
        },
    },
    # ------------------------------------------------------------------ BLOCK: 1 dòng / địa điểm, 1 khối
    {
        "source_table": "poi_address", "target_schema": _S, "target_table": "slv_3p_poi_address",
        "key": "poi_id",
        "columns": {
            "poi_id":           POI_ID,
            "address_line1":    _str("address_line1"),            # dòng địa chỉ 1 (mẫu: phường, quận, thành phố)
            "address_line2":    _str("address_line2"),            # dòng địa chỉ 2 (mẫu: "Đông Ngạc, Hà Nội")
            "house_number":     _str("house_number"),             # [THÊM 04/10] số nhà (17.831 / 21.168 có)
            "street":           _str("street"),                   # [THÊM 04/10] tên đường (19.475 có)
            "ward":             _str("ward"),                     # phường / xã
            "district":         _str("district"),                 # quận / huyện
            "city":             _str("city"),                     # thành phố
            "state_province":   _str("state_province"),           # tỉnh / thành phố trực thuộc TW
            "country":          _str("country"),                  # [THÊM 04/10] tên quốc gia, vd Vietnam
            "country_code":     _str("country_code"),             # mã quốc gia, vd VN
            "postal_code":      _str("postal_code"),              # mã bưu chính (13.255 có)
            "short_address":    _str("short_address"),            # địa chỉ ngắn (V12: 1.693 dòng là chuỗi rỗng)
            "full_address":     _str("full_address"),             # địa chỉ đầy đủ (V12: = address_google 100%)
            # [BỎ 04/10 D5] address_google (= full_address)
            "address_viet_map": _str("address_viet_map"),         # địa chỉ theo Vietmap (nghĩa chưa xác nhận; mẫu khác
                                                                  # full_address)
            "vietmap_status":   _str("vietmap_status"),           # [THÊM 04/10] kết quả gọi Vietmap, vd api_success
                                                                  # (17.209 có; nghĩa chưa xác nhận)
            "vietmap_message":  _off(_str("vietmap_message")),    # [TẮT 04/10 D1] thông báo của Vietmap
            "lat":              {"path": "lat", "type": "double"},   # vĩ độ
            "lng":              {"path": "lng", "type": "double"},   # kinh độ
            # [BỎ 04/10] geohash, h3_index_r8: không có trong payload (tính ở lớp sau từ lat/lng)
            "plus_code":        _str("plus_code"),                # Google Plus Code, vd 7P28QM74+79
            "timezone":         _str("timezone"),                 # múi giờ IANA, vd Asia/Ho_Chi_Minh
            # [BỎ 04/10 D2] address_json (cả khối, trùng các cột trên)
        },
    },
    {
        "source_table": "poi_contact", "target_schema": _S, "target_table": "slv_3p_poi_contact",
        "key": "poi_id",
        "columns": {
            "poi_id":              POI_ID,
            "phone":               _str("phone"),                 # điện thoại dạng trong nước, vd 0392 792 210
            "phone_raw":           _str("phone_raw"),             # [THÊM 04/10] điện thoại dạng quốc tế, vd +84 392 792 210
            "website_url":         _str("website_url"),           # website (12.521 có)
            "google_maps_url":     _str("google_maps_url"),       # [THÊM 04/10] link Google Maps của địa điểm
            "booking_links_json":  _off(_str("booking_links")),   # [TẮT 04/10 D1] link đặt chỗ
            "delivery_links_json": _off(_str("delivery_links")),  # [TẮT 04/10 D1] link giao hàng
            "social_links_json":   _off(_str("social_links")),    # [TẮT 04/10 D1] link mạng xã hội
            # [BỎ 04/10] hotline, email: không có trong payload. booking_* (5), social_* (3): cấu trúc con NB_20 đoán
            # [BỎ 04/10 D2] contact_json (cả khối)
        },
    },
    {
        "source_table": "poi_price", "target_schema": _S, "target_table": "slv_3p_poi_price",
        "key": "poi_id",
        "columns": {
            "poi_id":      POI_ID,
            "currency":    _str("currency"),
            "price_level": {"path": "price_level", "type": "int"},
            "price_min":   {"path": "price_min", "type": "decimal(18,2)"},
            "price_max":   {"path": "price_max", "type": "decimal(18,2)"},
        },
    },
    # [BỎ 09/10] slv_3p_poi_raw_data, slv_3p_poi_content, slv_3p_poi_enrichment, slv_3p_poi_review_i18n
    {
        "source_table": "poi_opening_hours", "target_schema": _S, "target_table": "slv_3p_poi_opening_hours",
        "key": "poi_id",
        "columns": {
            "poi_id":                POI_ID,
            "is_24_7":               _bool("is_24_7"),             # mở cửa 24/7
            "is_temporarily_closed": _bool("is_temporarily_closed"),   # [THÊM 04/10] tạm đóng cửa (V1: chỉ có false)
            "periods_json":          _str("periods"),              # mảng khung giờ mở cửa: day_of_week, open_hour,
                                                                   # open_minute, close_day, close_hour, close_minute (V10)
            "secondary_hours_json":  _str("secondary_hours"),      # [THÊM 04/10] giờ phụ theo loại (vd KITCHEN): mảng
                                                                   # {type, periods[], weekday_text[]}, nhiều tầng. Giữ khối
            "weekday_text_json":     _str("weekday_text"),         # [THÊM 04/10] mảng chuỗi hiển thị giờ theo ngày (≤ 7)
            "open_now":              _bool("open_now"),            # [THÊM 09/10] trường 1:1 trong poi_opening_hours
            # [BỎ 04/10] holiday_exceptions_json, created_at: không có trong payload
            # [BỎ 04/10 D2] opening_hours_json (cả khối). [SỬA 09/10] open_now tách thành cột boolean.
        },
    },
    {
        "source_table": "policies", "target_schema": _S, "target_table": "slv_3p_poi_policy",
        "key": "poi_id",
        "columns": {                                              # [TẮT 04/10 D1] cả bảng: khối luôn null
            "poi_id":        POI_ID,                              # (ctrl_mng_pipeline_config.is_active = 0)
            "policies_json": _str("$"),                           # chính sách của địa điểm (nghĩa chưa xác nhận)
        },
    },
    {
        "source_table": "poi_rating", "target_schema": _S, "target_table": "slv_3p_poi_rating",
        "key": "poi_id",
        "columns": {
            "poi_id":                POI_ID,
            "rating_overall":        {"path": "rating_overall", "type": "decimal(4,2)"},   # điểm đánh giá trung bình
            "rating_count":          {"path": "rating_count", "type": "bigint"},           # số lượt đánh giá
            "rating_breakdown_json": _off(_str("rating_breakdown")),  # [TẮT 04/10 D1] phân bố điểm
            "sub_ratings_json":      _off(_str("sub_ratings")),       # [TẮT 04/10 D1] điểm thành phần
            # [BỎ 04/10] source, rating_label, rating_weighted, last_review_at: không có trong payload
            # [BỎ 04/10 D2] rating_json (cả khối)
        },
    },
    {
        "source_table": "poi_amenity", "target_schema": _S, "target_table": "slv_3p_poi_amenity",
        "key": "poi_id",
        "columns": {
            "poi_id":              POI_ID,
            "amenity_schema_json": _str("amenity_schema"),         # JSON nhiều tầng: 1 khoá cấp 1 / tài liệu (facilities
                                                                   # 16.018, store_profile 2.826, center_profile 2.324),
                                                                   # giá trị là object (V3b). Nghĩa chưa xác nhận. Giữ JSON
            "ext_attributes_json": _str("ext_attributes"),         # [SỬA 09/10] object 1:1. E1: mọi giá trị đã thấy là string.
                                                                   # Schema gộp là union khoá, không phải khoá có trên mọi dòng.
                                                                   # Giữ JSON. Không map sang cờ nghiệp vụ.
            # [BỎ 04/10] amenities_json: khoá không tồn tại
            # [BỎ 04/10 D2] primary_type (= ext_attributes.primaryType, 18.896 / 19.046 có — V2): lấy từ
            #               ext_attributes_json ở transform ; amenity_json (cả khối, trùng các cột trên)
        },
    },
    # ------------------------------------------------------------------ CHILD: nhiều dòng / địa điểm
    {
        "source_table": "poi_review", "target_schema": _S, "target_table": "slv_3p_poi_review",
        "key": "review_id",
        "columns": {                                              # khớp payload (5 khoá), không đổi
            # = source_review_id cũ (NB_20): sha2(concat(source_name, source_id, author_name, time), 256)
            "review_id":    {"path": "_doc.source_name,_doc.source_id,author_name,time", "type": "string",
                             "rule": "HASH_SHA256"},
            "poi_id":       POI_ID,
            "author_name":  _str("author_name"),                  # tên người đánh giá
            "rating":       {"path": "rating", "type": "decimal(4,2)"},   # điểm của review
            "source":       _str("source"),                       # nguồn review, vd google
            "text":         _str("text"),                         # nội dung review (gốc)
            "time":         _str("time"),                         # thời điểm review, chuỗi gốc (thành phần khoá)
            "published_at": {"path": "time", "type": "timestamp"},   # thời điểm review, kiểu timestamp
        },
    },
    {
        "source_table": "poi_media", "target_schema": _S, "target_table": "slv_3p_poi_media",
        "key": "media_dedup_key",
        "columns": {
            # giống NB_20: sha2(concat_ws('|', trim(source_name), trim(source_id), trim(photo_api_uri)), 256)
            "media_dedup_key":   {"path": "_doc.source_name,_doc.source_id,photo_api_uri", "type": "string",
                                  "rule": "HASH_SHA256_PIPE"},
            "poi_id":            POI_ID,
            "media_id":          _str("media_id"),                # id ảnh của collector
            "photo_api_uri":     _str("photo_api_uri"),           # đường dẫn ảnh trong Google Places API (thành phần khoá)
            "media_type":        _str("media_type"),              # loại media, vd photo
            "blob_url":          _str("blob_url"),                # bản lưu trên Azure Blob (101.606 / 102.643 có)
            "original_url":      _str("original_url"),            # URL ảnh gốc (Google Places)
            "thumbnail_url":     _str("thumbnail_url"),           # URL ảnh thu nhỏ
            "processing_status": _str("processing_status"),       # trạng thái xử lý, vd active (nghĩa chưa xác nhận)
            "category":          _str("category"),                # nhóm ảnh, vd exterior (20.742 có)
            "caption":           _off(_str("caption")),           # [TẮT 04/10 D1] chú thích ảnh
            "is_blessed":        _bool("is_blessed"),             # cờ ảnh được chọn (nghĩa chưa xác nhận)
            "display_order":     {"path": "display_order", "type": "int"},   # thứ tự hiển thị
            "photographer":      _str("photographer"),            # tên người chụp / đăng ảnh
            "license":           _str("license"),                 # giấy phép, vd google_maps
            "file_size_bytes":   {"path": "file_size_bytes", "type": "bigint"},   # [THÊM 04/10] dung lượng file (21.402 có)
            "width_px":          _off({"path": "width_px", "type": "int"}),       # [TẮT 04/10 D1] chiều rộng ảnh
            "height_px":         _off({"path": "height_px", "type": "int"}),      # [TẮT 04/10 D1] chiều cao ảnh
            # [BỎ 04/10] media_fingerprint, source_field, source, created_at, updated_at: không có trong payload
        },
    },
    # [BỎ 09/10] hết slv_3p_poi_enrichment và slv_3p_poi_review_i18n
]


In [ ]:
# [THÊM 03/10 - 3P] Seed registry 3rd-party: thay toàn bộ dòng của lh_vv_bronze.dbo.brz_3rd_crawler_poi_stream (replaceWhere)
# Kỳ vọng [SỬA 10/10]: 10 bảng slv_3p_poi_*, 98 cột (15 cột tắt). policy tắt ở pipeline_config.
#          Khoá: poi_id (HASH_MD5_UUID); review review_id (HASH_SHA256); media media_dedup_key (HASH_SHA256_PIPE).
seed_registry(TABLE_CONFIGS_3P, "lh_vv_bronze.dbo", "brz_3rd_crawler_poi_stream")
show_registry("lh_vv_bronze.dbo", "brz_3rd_crawler_poi_stream")


Tạo các bảng log & debug


In [ ]:
%%sql
-- [SỬA 03/10] chỉ sửa COMMENT (read_mode, read_note) và đánh số mục. Cấu trúc 4 bảng giữ nguyên, khớp notebook partner
-- [SỬA 03/10 - 3P] chỉ sửa COMMENT: ý nghĩa cột với nguồn 3rd-party (brz_3rd_crawler_poi_stream). Cấu trúc không đổi
-- 1. ctrl_log_run — 1 dòng / 1 lần chạy notebook

CREATE TABLE IF NOT EXISTS lh_vv_ctrl.dbo.ctrl_log_run (
    exec_id             STRING    COMMENT 'ID mỗi lần notebook chạy (uuid) - khoá',
    run_id              STRING    COMMENT 'RunId của pipeline (chạy tay: = exec_id)',
    pl_name             STRING    COMMENT 'Pipeline, khớp ctrl_mng_pipeline_config.pl_name',
    nb_name             STRING    COMMENT 'Notebook thực thi',
    src_schema          STRING    COMMENT 'Schema bảng nguồn raw, vd lh_vv_bronze.dbo',
    src_tbl             STRING    COMMENT 'Bảng nguồn raw, vd partner_raw_data',
    watermark_id        STRING    COMMENT 'Khớp ctrl_mng_watermark.watermark_id',
    run_mode            STRING    COMMENT 'INCREMENTAL | FULL_RELOAD | DRY_RUN',
    table_filter        STRING    COMMENT 'Danh sách trg_tbl khi chạy lại một phần, NULL = tất cả',
    src_version_from    BIGINT    COMMENT 'last_src_version trước khi chạy',
    src_version_to      BIGINT    COMMENT 'Delta version snapshot đã đọc',
    watermark_from      TIMESTAMP COMMENT 'watermark_value trước khi chạy',
    watermark_to        TIMESTAMP COMMENT 'Max watermark_column của phần đã đọc (partner: EventProcessedUtcTime, 3rd-party: crawled_at)',
    overlap_minutes     INT       COMMENT 'Số phút đọc lùi so với watermark_from',
    read_rows           BIGINT    COMMENT 'Số event đọc được',
    valid_rows          BIGINT    COMMENT 'Số event hợp lệ đưa vào xử lý',
    ignored_rows        BIGINT    COMMENT 'Số event bỏ qua (bảng chưa cấu hình, tombstone, truncate...)',
    rejected_rows       BIGINT    COMMENT 'Số event lỗi ghi vào ctrl_cdc_reject',
    ignored_detail      STRING    COMMENT 'JSON số event bỏ qua theo src_object/lý do',
    tbl_total_count     INT       COMMENT 'Số bảng đích trong phạm vi chạy',
    tbl_success_count   INT       COMMENT 'Số bảng SUCCESS',
    tbl_failed_count    INT       COMMENT 'Số bảng FAILED',
    tbl_no_data_count   INT       COMMENT 'Số bảng không có event',
    status              STRING    COMMENT 'RUNNING | SUCCESS | PARTIAL_FAILED | FAILED | NO_DATA | SKIPPED_CONCURRENT | ABANDONED',
    started_at          TIMESTAMP COMMENT 'Bắt đầu (UTC)',
    ended_at            TIMESTAMP COMMENT 'Kết thúc (UTC)',
    duration_ms         BIGINT    COMMENT 'Thời gian chạy (ms)',
    run_params          STRING    COMMENT 'JSON tham số notebook',
    error_message       STRING    COMMENT 'Lỗi tổng (tóm tắt)',
    read_mode           STRING    COMMENT 'Cách đọc raw: VERSION | FULL',
    read_note           STRING    COMMENT 'Chi tiết cách đọc: khoảng version, số file, lý do đọc FULL',
    -- [SỬA 08/10] cột gold-log; extract để NULL
    output_versions_json STRING    COMMENT 'NB_00: version các bảng pl sở hữu khi run SUCCESS / NO_DATA (lát cắt cho luồng khác)'
) USING DELTA
COMMENT 'Log mỗi lần chạy notebook extract CDC';

-- 2. ctrl_log_table_run — 1 dòng / bảng đích / lần chạy
CREATE TABLE IF NOT EXISTS lh_vv_ctrl.dbo.ctrl_log_table_run (
    exec_id             STRING    COMMENT 'Khớp ctrl_log_run.exec_id',
    run_id              STRING    COMMENT 'RunId của pipeline',
    pl_name             STRING    COMMENT 'Pipeline',
    src_schema          STRING    COMMENT 'Schema bảng nguồn raw',
    src_tbl             STRING    COMMENT 'Bảng nguồn raw',
    src_object          STRING    COMMENT 'Bảng trong Debezium source.table',
    trg_schema          STRING    COMMENT 'Schema bảng đích, vd lh_vv_silver.dbo',
    trg_tbl             STRING    COMMENT 'Bảng đích, vd slv_products',
    priority            INT       COMMENT 'Wave chạy (ctrl_mng_pipeline_config.priority)',
    attempt_no          INT       COMMENT 'Lần thử (1 = lần đầu, >1 = retry)',
    input_rows          BIGINT    COMMENT 'Số event của bảng trong phần đã đọc (3rd-party: số dòng sau khi tách khối/mảng)',
    stale_rows          BIGINT    COMMENT 'Số event cũ hơn state, bị bỏ qua',
    applicable_rows     BIGINT    COMMENT 'Số event mới hơn state, được áp dụng',
    entity_rows         BIGINT    COMMENT 'Số entity (khoá) đưa vào MERGE (3rd-party: số khoá bảng)',
    upsert_event_rows   BIGINT    COMMENT 'Số event c/r/u được áp dụng',
    delete_event_rows   BIGINT    COMMENT 'Số event d được áp dụng',
    inserted_rows       BIGINT    COMMENT 'MERGE: số dòng insert',
    updated_rows        BIGINT    COMMENT 'MERGE: số dòng update (gồm xoá mềm)',
    cast_null_rows      BIGINT    COMMENT 'Số giá trị có ở raw nhưng thành NULL sau chuyển kiểu',
    rejected_rows       BIGINT    COMMENT 'Số event của bảng bị reject',
    status              STRING    COMMENT 'SUCCESS | FAILED | NO_DATA | SKIPPED',
    started_at          TIMESTAMP COMMENT 'Bắt đầu (UTC)',
    ended_at            TIMESTAMP COMMENT 'Kết thúc (UTC)',
    duration_ms         BIGINT    COMMENT 'Thời gian xử lý bảng (ms)',
    error_message       STRING    COMMENT 'Lỗi của bảng (tóm tắt)',
    -- [SỬA 08/10] cột gold-log; extract để NULL
    src_versions_json   STRING    COMMENT 'Node: JSON {bảng đầu vào: {v, tid, ts, dirty}} đã đọc',
    trg_version         BIGINT    COMMENT 'Node: version bảng đích sau MERGE',
    deactivated_rows    BIGINT    COMMENT 'Node: số dòng xoá mềm',
    qg_json             STRING    COMMENT 'Node: kết quả exit QG, hộp thư, cảnh báo'
) USING DELTA
COMMENT 'Log từng bảng đích trong mỗi lần chạy extract CDC';

-- 3. ctrl_cdc_state — 1 dòng / 1 entity / 1 bảng đích; khoá (trg_schema, trg_tbl, entity_key)
CREATE TABLE IF NOT EXISTS lh_vv_ctrl.dbo.ctrl_cdc_state (
    trg_schema          STRING    COMMENT 'Schema bảng đích (khoá)',
    trg_tbl             STRING    COMMENT 'Bảng đích (khoá)',
    entity_key          STRING    COMMENT 'Giá trị khoá; khoá ghép nối bằng | (khoá). 3rd-party: poi_id cho mọi bảng',
    src_schema          STRING    COMMENT 'Schema bảng nguồn raw',
    src_tbl             STRING    COMMENT 'Bảng nguồn raw',
    src_object          STRING    COMMENT 'Bảng trong Debezium source.table',
    cdc_op              STRING    COMMENT 'Op của event mới nhất: c | r | u | d',
    cdc_ts_ms           BIGINT    COMMENT 'source.ts_ms của event mới nhất (3rd-party: crawled_at tính ms)',
    cdc_lsn             BIGINT    COMMENT 'source.lsn của event mới nhất',
    raw_ingested_at     TIMESTAMP COMMENT 'EventProcessedUtcTime của event mới nhất (3rd-party: crawled_at)',
    is_deleted          BOOLEAN   COMMENT 'Entity đang ở trạng thái xoá mềm',
    src_version         BIGINT    COMMENT 'Delta version raw của lần áp dụng',
    last_run_id         STRING    COMMENT 'run_id lần cập nhật gần nhất',
    updated_at          TIMESTAMP COMMENT 'Thời điểm cập nhật state (UTC)'
) USING DELTA
PARTITIONED BY (src_tbl) -- FL_00 ForEach chạy tuần tự. Partition theo src_tbl để MERGE của partner và 3P (chạy tay hoặc pipeline khác) chỉ đọc partition của nguồn mình
COMMENT 'State CDC theo entity - đảm bảo chạy lại không áp dụng event cũ';

-- 4. ctrl_cdc_reject — 1 dòng / 1 event lỗi (không trùng nhờ event_hash)
CREATE TABLE IF NOT EXISTS lh_vv_ctrl.dbo.ctrl_cdc_reject (
    event_hash          STRING    COMMENT 'sha2(source||op||before||after) - khoá MERGE',
    pl_name             STRING    COMMENT 'Pipeline',
    src_schema          STRING    COMMENT 'Schema bảng nguồn raw',
    src_tbl             STRING    COMMENT 'Bảng nguồn raw',
    src_object          STRING    COMMENT 'Bảng trong Debezium source.table',
    src_db              STRING    COMMENT 'Database trong Debezium source.db (3rd-party: source_name)',
    trg_schema          STRING    COMMENT 'Schema bảng đích (nếu xác định được)',
    trg_tbl             STRING    COMMENT 'Bảng đích (nếu xác định được)',
    cdc_op              STRING    COMMENT 'Op của event',
    entity_key          STRING    COMMENT 'Khoá entity (có thể NULL nếu lỗi thiếu khoá)',
    cdc_ts_ms           BIGINT    COMMENT 'source.ts_ms',
    cdc_lsn             BIGINT    COMMENT 'source.lsn',
    raw_ingested_at     TIMESTAMP COMMENT 'EventProcessedUtcTime',
    reject_reason       STRING    COMMENT 'MISSING_ENTITY_KEY | MISSING_EVENT_ORDER | MISSING_RAW_CURSOR | INVALID_PAYLOAD',
    reject_detail       STRING    COMMENT 'Mô tả chi tiết',
    before_payload      STRING    COMMENT 'Giá trị before gốc',
    after_payload       STRING    COMMENT 'Giá trị after gốc (3rd-party: normalized_payload, hoặc phần tử lỗi của 1 bảng)',
    source_payload      STRING    COMMENT 'Giá trị source gốc (3rd-party: JSON event_id, source_name, source_id, language_code, crawled_at)',
    first_run_id        STRING    COMMENT 'run_id lần đầu phát hiện',
    last_run_id         STRING    COMMENT 'run_id lần gần nhất phát hiện',
    reject_count        INT       COMMENT 'Số lần bị đọc lại và reject',
    first_rejected_at   TIMESTAMP COMMENT 'Lần đầu reject (UTC)',
    last_rejected_at    TIMESTAMP COMMENT 'Lần gần nhất reject (UTC)',
    is_resolved         BOOLEAN   COMMENT 'Đã xử lý xong (đánh dấu tay hoặc khi nạp lại)',
    resolved_at         TIMESTAMP COMMENT 'Thời điểm đánh dấu đã xử lý'
) USING DELTA
PARTITIONED BY (src_tbl) -- FL_00 ForEach chạy tuần tự. Partition theo src_tbl để MERGE của partner và 3P (chạy tay hoặc pipeline khác) chỉ đọc partition của nguồn mình
COMMENT 'Event CDC lỗi, không áp dụng được vào bảng đích';


In [ ]:
# [SỬA 08/10] Bảng log đã có thì CREATE IF NOT EXISTS không thêm cột.
# Chỉ thêm 5 cột gold-log còn thiếu. Không seed ref, không cạnh, không watermark.
GOLD_LOG_COLUMNS = (
    ("lh_vv_ctrl.dbo.ctrl_log_run", (
        ("output_versions_json", "STRING", "NB_00: version các bảng pl sở hữu khi run SUCCESS / NO_DATA (lát cắt cho luồng khác)"),
    )),
    ("lh_vv_ctrl.dbo.ctrl_log_table_run", (
        ("src_versions_json", "STRING", "Node: JSON {bảng đầu vào: {v, tid, ts, dirty}} đã đọc"),
        ("trg_version", "BIGINT", "Node: version bảng đích sau MERGE"),
        ("deactivated_rows", "BIGINT", "Node: số dòng xoá mềm"),
        ("qg_json", "STRING", "Node: kết quả exit QG, hộp thư, cảnh báo"),
    )),
)
for tbl, cols in GOLD_LOG_COLUMNS:
    have = {f.name.lower() for f in spark.table(tbl).schema.fields}
    miss = [c for c in cols if c[0].lower() not in have]
    if not miss:
        print(f"{tbl}: đã đủ cột gold-log")
        continue
    ddl = ", ".join(f"{n} {t} COMMENT '{c}'" for n, t, c in miss)
    spark.sql(f"ALTER TABLE {tbl} ADD COLUMNS ({ddl})")
    print(f"{tbl}: đã thêm {', '.join(n for n, _, _ in miss)}")


Kiểm tra cấu hình


In [ ]:
%%sql
-- D1. Trùng bảng đích trong pipeline_config
SELECT pl_name, trg_schema, trg_tbl, COUNT(*) AS cnt
FROM lh_vv_ctrl.dbo.ctrl_mng_pipeline_config
GROUP BY pl_name, trg_schema, trg_tbl
HAVING COUNT(*) > 1;
 
-- D2. Bảng partner có trong pipeline_config nhưng chưa có mapping cột
SELECT p.trg_schema, p.trg_tbl
FROM lh_vv_ctrl.dbo.ctrl_mng_pipeline_config p
LEFT ANTI JOIN lh_vv_ctrl.dbo.ctrl_cfg_schema_registry r
  ON r.src_schema = p.src_schema AND r.src_tbl = p.src_tbl
 AND r.trg_schema = p.trg_schema AND r.trg_tbl = p.trg_tbl
WHERE p.src_tbl = 'partner_raw_data' AND p.is_active = 1;
 
-- D3. Mapping cột không có bảng tương ứng trong pipeline_config
SELECT DISTINCT r.src_tbl, r.trg_schema, r.trg_tbl
FROM lh_vv_ctrl.dbo.ctrl_cfg_schema_registry r
LEFT ANTI JOIN lh_vv_ctrl.dbo.ctrl_mng_pipeline_config p
  ON p.src_schema = r.src_schema AND p.src_tbl = r.src_tbl
 AND p.trg_schema = r.trg_schema AND p.trg_tbl = r.trg_tbl;

-- [THÊM 03/10] D4. Tiền tố bảng đích theo nguồn.
--   Kỳ vọng: pipeline_config partner_raw_data = slv_pn_ (23 bảng), brz_3rd_crawler_poi_stream = slv_3p_poi_ (14 bảng),
--            schema_registry partner_raw_data = slv_pn_ (23 bảng, 301 cột). Không còn dòng prefix slv_
--   [SỬA 03/10 - 3P] thêm: schema_registry brz_3rd_crawler_poi_stream = slv_3p_ (14 bảng, 182 cột). Regex tiền tố
--             slv_3p_poi_ -> slv_3p_ (bảng gốc slv_3p_poi không có dấu _ cuối)
SELECT 'pipeline_config' AS ctrl_tbl, src_tbl,
       regexp_extract(trg_tbl, '^(slv_pn_|slv_3p_|slv_)', 1) AS prefix,
       COUNT(DISTINCT trg_tbl) AS so_bang, COUNT(*) AS so_dong
FROM lh_vv_ctrl.dbo.ctrl_mng_pipeline_config
GROUP BY 1, 2, 3
UNION ALL
SELECT 'schema_registry', src_tbl,
       regexp_extract(trg_tbl, '^(slv_pn_|slv_3p_|slv_)', 1),
       COUNT(DISTINCT trg_tbl), COUNT(*)
FROM lh_vv_ctrl.dbo.ctrl_cfg_schema_registry
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3;

-- [THÊM 03/10] D5. Mỗi nguồn active có đúng 1 dòng watermark cấp nguồn (trg_tbl NULL).
--   Kỳ vọng: so_dong_wm = 1, partner status INITIALIZED, last_src_version NULL
SELECT p.src_schema, p.src_tbl, COUNT(w.watermark_id) AS so_dong_wm,
       MAX(w.status) AS status, MAX(w.last_src_version) AS last_src_version
FROM (SELECT DISTINCT src_schema, src_tbl FROM lh_vv_ctrl.dbo.ctrl_mng_pipeline_config WHERE is_active = 1) p
LEFT JOIN lh_vv_ctrl.dbo.ctrl_mng_watermark w
  ON w.src_schema = p.src_schema AND w.src_tbl = p.src_tbl AND w.trg_tbl IS NULL
GROUP BY 1, 2;

-- [THÊM 03/10] D6. Wave (priority). Kỳ vọng: partner 1 = 12 bảng, 2 = 11 bảng.
--   [SỬA 03/10 - 3P] brz_3rd_crawler_poi_stream: 1 = 14 bảng (bản trước: 1 = 1, 2 = 11, 3 = 2 theo luồng cũ)
--   [SỬA 09/10 - 3P] brz_3rd_crawler_poi_stream: 9 bảng (8 active, policy is_active = 0)
SELECT src_tbl, priority, COUNT(*) AS so_bang
FROM lh_vv_ctrl.dbo.ctrl_mng_pipeline_config
GROUP BY 1, 2
ORDER BY 1, 2;

-- [THÊM 03/10 - 3P] D7. Cấu hình bảng 3rd-party (pipeline_config + registry).
--   Kỳ vọng: 9 dòng. is_active = 0 chỉ ở policy. Không còn price, raw_data, content, enrichment, review_i18n.
--            dedup_order chỉ ở media. so_cot_tat = số cột active = false
SELECT p.trg_tbl, p.load_mode, p.align_path, p.dedup_order, p.priority, p.is_active,
       MAX(r.src_object) AS src_object, COUNT(r.trg_column) AS so_cot,
       SUM(CASE WHEN r.is_active THEN 0 ELSE 1 END) AS so_cot_tat,
       MAX(CASE WHEN r.trg_column = 'poi_id' THEN true ELSE false END) AS co_poi_id,
       array_join(array_sort(collect_list(CASE WHEN r.is_key THEN r.trg_column END)), ', ') AS khoa
FROM lh_vv_ctrl.dbo.ctrl_mng_pipeline_config p
LEFT JOIN lh_vv_ctrl.dbo.ctrl_cfg_schema_registry r
  ON r.src_schema = p.src_schema AND r.src_tbl = p.src_tbl AND r.trg_schema = p.trg_schema AND r.trg_tbl = p.trg_tbl
WHERE p.src_tbl = 'brz_3rd_crawler_poi_stream'
GROUP BY 1, 2, 3, 4, 5, 6
ORDER BY 1;


In [ ]:
%%sql
-- [THÊM 03/10] K1. Khoá order_item_* có làm mất dòng không (đọc toàn bộ partner_raw_data, chạy vài phút).
-- n_* = số giá trị khác nhau trong after. Khoá đang dùng có ít giá trị hơn khoá chi tiết -> MERGE gộp nhiều dòng thành 1.
--   order_item_tickets : khoá order_ref_id
--   order_item_hotels  : khoá order_id, n_order_ref_id > n_order_id -> phải đổi khoá
--   order_item_flights : khoá order_id, n_ref_segment > n_order_id  -> đổi khoá sang ["order_ref_id", "segment_id"]
SELECT get_json_object(source, '$.table')                                    AS src_object,
       COUNT(*)                                                               AS events,
       COUNT(DISTINCT get_json_object(after, '$.order_id'))                   AS n_order_id,
       COUNT(DISTINCT get_json_object(after, '$.order_ref_id'))               AS n_order_ref_id,
       COUNT(DISTINCT concat_ws('|', get_json_object(after, '$.order_ref_id'),
                                     get_json_object(after, '$.segment_id')))  AS n_ref_segment
FROM lh_vv_bronze.dbo.partner_raw_data
WHERE get_json_object(source, '$.table') IN ('order_item_tickets', 'order_item_hotels', 'order_item_flights')
  AND op IN ('c', 'r', 'u')
GROUP BY 1
ORDER BY 1;
